In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV,cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [3]:
data = load_breast_cancer()
df = pd.DataFrame(data.data,columns=data.feature_names)
df["target"] = data.target
print(df.shape)
print(df.head())


(569, 31)
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0   

In [4]:
X = df.drop("target",axis=1)
y = df["target"]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [5]:
svm_pipeline = Pipeline([("scaler", StandardScaler()),("svm", SVC())])
svm_pipeline.fit(X_train, y_train)
svm_pred = svm_pipeline.predict(X_test)

In [6]:
print("Basic SVM Results")

print("Accuracy :", accuracy_score(y_test, svm_pred))
print("Precision:", precision_score(y_test, svm_pred))
print("Recall   :", recall_score(y_test, svm_pred))
print("F1 Score :", f1_score(y_test, svm_pred))

Basic SVM Results
Accuracy : 0.9824561403508771
Precision: 0.9861111111111112
Recall   : 0.9861111111111112
F1 Score : 0.9861111111111112


In [7]:
for kernel in ["linear","rbf","poly"] :
    model =Pipeline([("Scaler",StandardScaler()),("svm",SVC(kernel=kernel))])
    model.fit(X_train,y_train)
    pred = model.predict(X_test)
    accuracy = accuracy_score(y_test,pred)
    print(f"Kernel: {kernel}, Accuracy: {accuracy:.4f}")

Kernel: linear, Accuracy: 0.9737
Kernel: rbf, Accuracy: 0.9825
Kernel: poly, Accuracy: 0.9123


In [9]:
param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.01, 0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1]
}
grid = GridSearchCV(
    svm_pipeline,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best parameters: {'svm__C': 10, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Best CV score: 0.9802197802197803


In [10]:
best_svm = grid.best_estimator_
svm_final_pred = best_svm.predict(X_test)

print("Final SVM Results")

print("Accuracy :", accuracy_score(y_test, svm_final_pred))
print("Precision:", precision_score(y_test, svm_final_pred))
print("Recall   :", recall_score(y_test, svm_final_pred))
print("F1 Score :", f1_score(y_test, svm_final_pred))

Final SVM Results
Accuracy : 0.9824561403508771
Precision: 0.9861111111111112
Recall   : 0.9861111111111112
F1 Score : 0.9861111111111112
